# MLOps track: SageMaker job API and a Step Functions pipeline

You practise the operational contract around a model: registering training jobs, models and endpoints, and orchestrating a refresh with a quality gate.

**Note:** the emulator accepts these calls and tracks status, but runs no containers. Training does not happen and an endpoint cannot return predictions. What you learn here is the request shapes, IAM wiring, status handling and orchestration.

In [1]:
import sys, json, time, datetime as dt
sys.path.insert(0, "../..")          # repository root: emu.py, lab_common.py
import pandas as pd
import matplotlib.pyplot as plt
import emu
from lab_common import *

df = ensure_lake()                   # buckets, loan book in S3, Glue table (safe to re-run)
print("Connected to", emu.settings()["endpoint_url"], "-", len(df), "loans in the lake")

Connected to https://127.0.0.1:9297 - 2000 loans in the lake


## 1. Train locally and publish the artefact

In [2]:
import io, pickle, tarfile
model, auc, _, _ = train_default_model(df)
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w:gz") as tar:
    blob = pickle.dumps(model); info = tarfile.TarInfo("model.pkl"); info.size = len(blob)
    tar.addfile(info, io.BytesIO(blob))
emu.client("s3").put_object(Bucket=LAKE, Key="models/default-model/model.tar.gz", Body=buf.getvalue())
print(f"Hold-out AUC {auc:.3f}; artefact in s3://{LAKE}/models/default-model/model.tar.gz")

Hold-out AUC 0.789; artefact in s3://bank-lake/models/default-model/model.tar.gz


## 2. SageMaker: training job, model, endpoint

Each call needs an IAM role the service can assume; `ensure_role()` creates one.

In [3]:
sm, role_arn = emu.client("sagemaker"), ensure_role()
IMAGE = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
JOB = f"default-model-{int(time.time())}"

attempt("CreateTrainingJob", lambda: sm.create_training_job(
    TrainingJobName=JOB, RoleArn=role_arn,
    AlgorithmSpecification={"TrainingImage": IMAGE, "TrainingInputMode": "File"},
    InputDataConfig=[{"ChannelName": "train", "ContentType": "text/csv", "DataSource": {"S3DataSource": {
        "S3DataType": "S3Prefix", "S3Uri": f"s3://{LAKE}/loans/", "S3DataDistributionType": "FullyReplicated"}}}],
    OutputDataConfig={"S3OutputPath": f"s3://{LAKE}/training-output/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 5},
    StoppingCondition={"MaxRuntimeInSeconds": 600}))

if "default-model" not in [m["ModelName"] for m in sm.list_models()["Models"]]:
    attempt("CreateModel", lambda: sm.create_model(ModelName="default-model", ExecutionRoleArn=role_arn,
        PrimaryContainer={"Image": IMAGE, "ModelDataUrl": f"s3://{LAKE}/models/default-model/model.tar.gz"}))
def missing(describe):                 # ListEndpointConfigs is not implemented in the emulator, so describe instead
    try:
        describe(); return False
    except sm.exceptions.ClientError:
        return True

if missing(lambda: sm.describe_endpoint_config(EndpointConfigName="default-model-config")):
    attempt("CreateEndpointConfig", lambda: sm.create_endpoint_config(EndpointConfigName="default-model-config",
        ProductionVariants=[{"VariantName": "primary", "ModelName": "default-model", "InstanceType": "ml.m5.large", "InitialInstanceCount": 1}]))
if missing(lambda: sm.describe_endpoint(EndpointName="default-model-endpoint")):
    attempt("CreateEndpoint", lambda: sm.create_endpoint(EndpointName="default-model-endpoint", EndpointConfigName="default-model-config"))

time.sleep(3)
print("Training job:", sm.describe_training_job(TrainingJobName=JOB)["TrainingJobStatus"])
print("Endpoint    :", sm.describe_endpoint(EndpointName="default-model-endpoint")["EndpointStatus"])

OK      CreateTrainingJob
OK      CreateEndpointConfig
OK      CreateEndpoint


Training job: InProgress
Endpoint    : InService


### The limit: invoking the endpoint

The endpoint reports `InService`, but no container is behind it. The refusal below is expected and shows where the emulator stops.

In [4]:
attempt("InvokeEndpoint", lambda: emu.client("sagemaker-runtime").invoke_endpoint(
    EndpointName="default-model-endpoint", ContentType="text/csv", Body=b"690,50000,120000,0.38,36,0"))

REFUSED InvokeEndpoint: ModelError - The model artifacts at s3://bank-lake/models/default-model/model.tar.gz hold no model this emulator can compute an inference from: a SageMaker model runs as a container image, which this emulator does


## 3. Step Functions: a model-refresh pipeline with a quality gate

The state machine validates the data, checks AUC against a 0.70 gate, and approves or rejects. Run it once with the real AUC and once with a failing value.

In [5]:
sfn = emu.client("stepfunctions")
definition = {"Comment": "Monthly default-model refresh", "StartAt": "ValidateData", "States": {
    "ValidateData": {"Type": "Pass", "Result": {"rows": len(df)}, "ResultPath": "$.validation", "Next": "CheckQuality"},
    "CheckQuality": {"Type": "Choice", "Choices": [{"Variable": "$.auc", "NumericGreaterThanEquals": 0.70, "Next": "Approve"}], "Default": "Reject"},
    "Approve": {"Type": "Pass", "Result": "model approved for deployment", "ResultPath": "$.decision", "End": True},
    "Reject": {"Type": "Fail", "Error": "ModelQuality", "Cause": "AUC below the 0.70 gate"}}}
machines = {m["name"]: m["stateMachineArn"] for m in sfn.list_state_machines()["stateMachines"]}
sm_arn = machines.get("default-model-refresh") or sfn.create_state_machine(
    name="default-model-refresh", roleArn=role_arn, definition=json.dumps(definition))["stateMachineArn"]

def run_pipeline(auc_value):
    ex = sfn.start_execution(stateMachineArn=sm_arn, input=json.dumps({"auc": auc_value}))
    for _ in range(50):
        d = sfn.describe_execution(executionArn=ex["executionArn"])
        if d["status"] != "RUNNING":
            break
        time.sleep(0.2)
    return {"auc": auc_value, "status": d["status"], "result": d.get("output") or d.get("cause")}

pd.DataFrame([run_pipeline(round(auc, 3)), run_pipeline(0.61)])

,auc,status,result
0,0.789,SUCCEEDED,"{""auc"":0.789,""validation"":{""rows"":2000},""decis..."
1,0.610,FAILED,AUC below the 0.70 gate


In [6]:
runs = sfn.list_executions(stateMachineArn=sm_arn)["executions"]
pd.DataFrame([{"execution": r["name"][:8], "status": r["status"], "started": r["startDate"]} for r in runs])

,execution,status,started
0,3040ffd3,FAILED,2026-11-17 06:57:16.810000+00:00
1,7cab0477,SUCCEEDED,2026-11-17 06:57:16.177000+00:00
2,ab634333,FAILED,2026-11-17 06:56:46.409000+00:00
3,7543238b,SUCCEEDED,2026-11-17 06:56:45.769000+00:00


## Exercises

1. Add a `Wait` state before `CheckQuality` and a `Retry` policy, then re-run both paths.
2. Register the model in a model package group (`create_model_package_group`) and list it.
3. Publish the pipeline outcome to an SNS topic from the notebook after each run.